In [1]:
import ast, json, re

def strip_test_harness(code: str) -> str:
    """Remove def check(...), def main(...), top-level check(...)/main() calls,
    and `if __name__ == "__main__": ...` blocks from a code string.
    Keeps solution code (classes/functions) intact.
    """
    if code is None:
        return None

    s = str(code).strip().strip('`').strip()

    # If the whole thing looks like a JSON-escaped string, try to unescape it.
    if (len(s) >= 2) and ((s[0] == s[-1] == '"') or (s[0] == s[-1] == "'")):
        try:
            s = json.loads(s)
        except Exception:
            s = s[1:-1]

    # Try AST-based removal (preferred)
    try:
        tree = ast.parse(s)

        class HarnessStripper(ast.NodeTransformer):
            def visit_FunctionDef(self, node):
                if node.name in ("check", "main"):
                    return None
                return self.generic_visit(node)

            def visit_AsyncFunctionDef(self, node):
                if node.name in ("check", "main"):
                    return None
                return self.generic_visit(node)

            def visit_If(self, node):
                # Remove: if __name__ == "__main__": ...
                try:
                    cmp = node.test
                    if (
                        isinstance(cmp, ast.Compare)
                        and isinstance(cmp.left, ast.Name) and cmp.left.id == "__name__"
                        and any(isinstance(op, ast.Eq) for op in cmp.ops)
                        and any(isinstance(c, ast.Constant) and c.value == "__main__" for c in cmp.comparators)
                    ):
                        return None
                except Exception:
                    pass
                return self.generic_visit(node)

        new_tree = HarnessStripper().visit(tree)

        # Also drop *top-level* calls to main() or check(...)
        pruned_body = []
        for node in new_tree.body:
            # Expr that is a Call to main() or check(...)
            if isinstance(node, ast.Expr) and isinstance(node.value, ast.Call):
                func = node.value.func
                if isinstance(func, ast.Name) and func.id in {"main", "check"}:
                    continue
            pruned_body.append(node)
        new_tree.body = pruned_body
        ast.fix_missing_locations(new_tree)

        if hasattr(ast, "unparse"):
            s = ast.unparse(new_tree)
        else:
            # If running on very old Python (unlikely), fall back to regex below
            raise RuntimeError("ast.unparse not available")

    except Exception:
        # Regex fallback if code isn't valid Python or unparse not available
        patterns = [
            r"(?ms)^\s*def\s+check\s*\([^)]*\):\s*(?:\n(?:\s{4}|\t).*)*",          # def check(...)
            r"(?ms)^\s*def\s+main\s*\([^)]*\):\s*(?:\n(?:\s{4}|\t).*)*",           # def main(...)
            r"(?ms)^\s*if\s+__name__\s*==\s*['\"]__main__['\"]\s*:\s*(?:\n.*?)(?=^\S|\Z)",  # if __name__ ...
            r"(?m)^\s*check\s*\(.*\)\s*$",                                         # top-level check(...)
            r"(?m)^\s*main\s*\(\s*\)\s*$",                                         # top-level main()
        ]
        for p in patterns:
            s = re.sub(p, "", s)

    # Tidy extra blank lines
    s = re.sub(r"\n{3,}", "\n\n", s).strip() + "\n"
    return s


In [2]:
#!/usr/bin/env python3
import argparse, json, csv, sys
from pathlib import Path
import pandas as pd

def _as_path(p) -> Path:
    return p if isinstance(p, Path) else Path(p)

def load_solutions(sol_path: Path) -> pd.DataFrame:
    """
    Load solutions from:
      - JSON array: [{id, response, score?}, ...]
      - JSONL: one JSON object per line
      - Dict keyed by id: { "1": {"response": "...", "score": ...}, ... } or { "1": "..." }
    Returns DataFrame with columns: id, response, (optional) score
    """
    sol_path = _as_path(sol_path)  # <- fix: accept str or Path
    text = sol_path.read_text(encoding="utf-8").strip()

    records = None

    # Try JSON parse first
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            records = obj
        elif isinstance(obj, dict):
            # Dict keyed by id
            recs = []
            for k, v in obj.items():
                if isinstance(v, dict):
                    rec = {"id": k, **v}
                else:
                    rec = {"id": k, "response": v}
                recs.append(rec)
            records = recs
        else:
            raise ValueError("Top-level JSON must be list or dict")
    except json.JSONDecodeError:
        # Fallback to JSON Lines
        recs = []
        for line in text.splitlines():
            line = line.strip()
            if not line:
                continue
            recs.append(json.loads(line))
        records = recs

    # Normalize minimal fields
    norm = []
    for r in records:
        if not isinstance(r, dict):
            continue
        rid = r.get("id")
        # Handle id possibly nested as string/number
        if rid is None:
            # try to recover if dict keyed and not merged above
            continue
        resp = r.get("response")
        score = r.get("score", None)
        norm.append({"id": rid, "response": strip_test_harness(resp), "score": score})

    df = pd.DataFrame(norm)

    # Coerce id to string for robust joining against CSV (which may have id as str/int)
    df["id"] = df["id"].astype(str)

    # Some pipelines accidentally store response as non-string (e.g., float NaN); coerce to str safely
    def _to_str(x):
        if x is None:
            return None
        # avoid "nan" for floats meant to be missing
        if isinstance(x, float) and pd.isna(x):
            return None
        return str(x)
    df["response"] = df["response"].map(_to_str)

    # If duplicates: keep the highest score if present; else keep the last occurrence
    if "score" in df.columns and df["score"].notna().any():
        # For missing scores, treat as -inf so scored rows win
        df["_score_rank"] = df["score"].fillna(float("-inf"))
        df = df.sort_values(["id", "_score_rank"], ascending=[True, False])
        df = df.drop_duplicates(subset=["id"], keep="first").drop(columns=["_score_rank"])
    else:
        df = df.drop_duplicates(subset=["id"], keep="last")

    return df[["id", "response","score"]]



In [3]:



# Read dev CSV (keep id as string to be safe)
dev = pd.read_csv("dev_en_gemini.csv", dtype={"id": str}, encoding="utf-8")

# Load solutions and merge
sol = load_solutions("dev_results.json")
merged = dev.merge(sol, on="id", how="left")

# Write out; quoting ensures code/newlines in response are preserved
merged.to_csv("dev_with_sols.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)




In [4]:
import pandas as pd

df1 = pd.read_csv("trial_with_en_v1.csv")
df2 = pd.read_csv("dev_with_sols.csv")
# Combine datasets row-wise
merged = pd.concat([df1, df2], ignore_index=True)

# Assign new unique ids
merged.insert(0, "new_id", range(1, len(merged)+1))
merged["id"] = merged ["new_id"]
merged = merged.drop(columns=["new_id","score"])

print(merged.head())
merged.to_csv("rag.csv")


   id                                        instruction  \
0   1  প্রথম n সংখ্যার ক্ষুদ্রতম গুণিতক খুঁজে বের করা...   
1   2  সাধারণ কীগুলির জন্য মান যোগ করে দুটি অভিধানকে ...   
2   3  ১ থেকে এন পর্যন্ত মোট আনসেট বিট গণনা করার জন্য...   
3   4  একটি ফাংশন লিখুন যা প্রদত্ত সংখ্যাটি এমনকি হলে...   
4   5  দ্বিপদী সহগগুলির বর্গক্ষেত্রের যোগফল খুঁজে বের...   

                                            response  \
0  def smallest_multiple(n):\r\n    if (n<=2):\r\...   
1  from collections import Counter\r\ndef add_dic...   
2  def count_Unset_Bits(n) :  \r\n    cnt = 0;  \...   
3  def even_num(x):\r\n    if x%2==0:\r\n        ...   
4  def factorial(start,end): \r\n    res = 1 \r\n...   

                                           test_list  \
0  "['assert smallest_multiple(13)==360360', 'ass...   
1  "[\"assert add_dict({'a': 100, 'b': 200, 'c':3...   
2  "['assert count_Unset_Bits(2) == 1', 'assert c...   
3  "['assert even_num(13.5)==False', 'assert even...   
4  "['assert sum_of_sq